<a href="https://colab.research.google.com/github/Yousaf451/machine-learning-deployment-demo/blob/main/model_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================================================
# ADULT INCOME CLASSIFICATION PROJECT
# Ready-made version with exact dataset columns
# =========================================================

import os
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [2]:
# =========================================================
# 1) SETTINGS
# =========================================================

csv_path = "/content/adult.data"   # change this to your file name
test_size = 0.2
random_state = 42
model_file = "best_classification_model.joblib"
metadata_file = "model_metadata.joblib"


In [3]:
# =========================================================
# 2) LOAD DATA
# =========================================================

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"CSV file not found: {csv_path}")

# Adult Income dataset usually has no header row
column_names = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
    "native-country",
    "income"
]

df = pd.read_csv(
    csv_path,
    header=None,
    names=column_names,
    na_values="?"
)

print("✅ Data loaded successfully")
print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())


✅ Data loaded successfully
Shape: (32561, 15)

First 5 rows:
   age          workclass  fnlwgt   education  education-num  \
0   39          State-gov   77516   Bachelors             13   
1   50   Self-emp-not-inc   83311   Bachelors             13   
2   38            Private  215646     HS-grad              9   
3   53            Private  234721        11th              7   
4   28            Private  338409   Bachelors             13   

        marital-status          occupation    relationship    race      sex  \
0        Never-married        Adm-clerical   Not-in-family   White     Male   
1   Married-civ-spouse     Exec-managerial         Husband   White     Male   
2             Divorced   Handlers-cleaners   Not-in-family   White     Male   
3   Married-civ-spouse   Handlers-cleaners         Husband   Black     Male   
4   Married-civ-spouse      Prof-specialty            Wife   Black   Female   

   capital-gain  capital-loss  hours-per-week  native-country  income  
0      

In [4]:
# =========================================================
# 3) CLEAN DATA
# =========================================================

# Remove duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

# Strip spaces from all object columns
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].astype(str).str.strip()

# Clean target column
df["income"] = df["income"].str.replace(".", "", regex=False).str.strip()

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nTarget classes:")
print(df["income"].value_counts())


Missing values per column:
age               0
workclass         0
fnlwgt            0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
income            0
dtype: int64

Target classes:
income
<=50K    24698
>50K      7839
Name: count, dtype: int64


In [5]:
# =========================================================
# 4) SPLIT FEATURES AND TARGET
# =========================================================

X = df.drop(columns=["income"])
y = df["income"]

print("\nFeature shape:", X.shape)
print("Target shape:", y.shape)



Feature shape: (32537, 14)
Target shape: (32537,)


In [6]:
# =========================================================
# 5) IDENTIFY NUMERIC AND CATEGORICAL COLUMNS
# =========================================================

numeric_features = [
    "age",
    "fnlwgt",
    "education-num",
    "capital-gain",
    "capital-loss",
    "hours-per-week"
]

categorical_features = [
    "workclass",
    "education",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country"
]

print("\nNumeric features:", numeric_features)
print("Categorical features:", categorical_features)



Numeric features: ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
Categorical features: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']


In [7]:
# =========================================================
# 6) PREPROCESSING PIPELINES
# =========================================================

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

try:
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])
except TypeError:
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=False))
    ])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])


In [8]:
# =========================================================
# 7) TRAIN-TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=test_size,
    random_state=random_state,
    stratify=y
)

print("\nTrain shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)



Train shape: (26029, 14) (26029,)
Test shape: (6508, 14) (6508,)


In [9]:
# =========================================================
# 8) MODELS
# =========================================================

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Random Forest": RandomForestClassifier(
        random_state=random_state,
        n_estimators=200,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=random_state)
}

results = []
trained_pipelines = {}


In [10]:
# =========================================================
# 9) TRAIN + EVALUATE
# =========================================================

for model_name, model in models.items():
    print(f"\n🔹 Training: {model_name}")

    clf = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    results.append({
        "Model": model_name,
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    })

    trained_pipelines[model_name] = clf
    print(f"✅ {model_name} done")


🔹 Training: Logistic Regression
✅ Logistic Regression done

🔹 Training: Random Forest
✅ Random Forest done

🔹 Training: Gradient Boosting
✅ Gradient Boosting done


In [11]:
# =========================================================
# 10) COMPARE MODELS
# =========================================================

results_df = pd.DataFrame(results).sort_values(by="F1-score", ascending=False)

print("\n=================================================")
print("MODEL COMPARISON")
print("=================================================")
print(results_df)



MODEL COMPARISON
                 Model  Accuracy  Precision    Recall  F1-score
2    Gradient Boosting  0.872004   0.867312  0.872004  0.866409
1        Random Forest  0.859096   0.854304  0.859096  0.855601
0  Logistic Regression  0.857867   0.852465  0.857867  0.853604


In [12]:
# =========================================================
# 11) SELECT BEST MODEL
# =========================================================

best_model_name = results_df.iloc[0]["Model"]
best_model = trained_pipelines[best_model_name]

print(f"\n🏆 Best Model Selected: {best_model_name}")



🏆 Best Model Selected: Gradient Boosting


In [13]:
# =========================================================
# 12) FINAL EVALUATION
# =========================================================

y_pred_best = best_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred_best)
precision = precision_score(y_test, y_pred_best, average="weighted", zero_division=0)
recall = recall_score(y_test, y_pred_best, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred_best, average="weighted", zero_division=0)

print("\n=================================================")
print("FINAL EVALUATION")
print("=================================================")
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1-score :", round(f1, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best, zero_division=0))



FINAL EVALUATION
Accuracy : 0.872
Precision: 0.8673
Recall   : 0.872
F1-score : 0.8664

Confusion Matrix:
[[4692  248]
 [ 585  983]]

Classification Report:
              precision    recall  f1-score   support

       <=50K       0.89      0.95      0.92      4940
        >50K       0.80      0.63      0.70      1568

    accuracy                           0.87      6508
   macro avg       0.84      0.79      0.81      6508
weighted avg       0.87      0.87      0.87      6508



In [14]:
# =========================================================
# 13) SAVE MODEL
# =========================================================

joblib.dump(best_model, model_file)

metadata = {
    "dataset": "Adult Income",
    "target_column": "income",
    "feature_columns": X.columns.tolist(),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "best_model_name": best_model_name
}

joblib.dump(metadata, metadata_file)

print(f"\n✅ Model saved: {model_file}")
print(f"✅ Metadata saved: {metadata_file}")



✅ Model saved: best_classification_model.joblib
✅ Metadata saved: model_metadata.joblib


In [15]:
# =========================================================
# 14) RELOAD AND PREDICT EXAMPLE
# =========================================================

print("\n=================================================")
print("RELOAD EXAMPLE")
print("=================================================")

loaded_model = joblib.load(model_file)
loaded_metadata = joblib.load(metadata_file)

print("Loaded model:", loaded_metadata["best_model_name"])
print("Target column:", loaded_metadata["target_column"])

sample_data = X_test.head(5)
sample_predictions = loaded_model.predict(sample_data)

print("\nSample predictions on first 5 test rows:")
print(sample_predictions)


RELOAD EXAMPLE
Loaded model: Gradient Boosting
Target column: income

Sample predictions on first 5 test rows:
['<=50K' '>50K' '<=50K' '<=50K' '<=50K']
